In [2]:
!pip install wandb

   ---------------------------------------- 0.0/25.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/25.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/25.0 MB ? eta -:--:--
   -- ------------------------------------- 1.3/25.0 MB 4.0 MB/s eta 0:00:06
   --- ------------------------------------ 2.4/25.0 MB 4.1 MB/s eta 0:00:06
   ----- ---------------------------------- 3.7/25.0 MB 4.8 MB/s eta 0:00:05
   ------- -------------------------------- 5.0/25.0 MB 5.0 MB/s eta 0:00:04
   ---------- ----------------------------- 6.6/25.0 MB 5.4 MB/s eta 0:00:04
   ------------- -------------------------- 8.1/25.0 MB 5.8 MB/s eta 0:00:03
   --------------- ------------------------ 9.7/25.0 MB 6.0 MB/s eta 0:00:03
   ------------------ --------------------- 11.8/25.0 MB 6.4 MB/s eta 0:00:03
   ---------------------- ----------------- 13.9/25.0 MB 6.8 MB/s eta 0:00:02
   -------------------------- ------------- 16.3/25.0 MB 7.2 MB/s eta 0:00:02
   --------------

# # Проект: сравнение моделей классификации с wandb

# ## 1. Загрузка данных
Использую встроенный датасет breast cancer из sklearn.

## 2. Обучение Logistic Regression
Обучаю первую модель и логирую метрики в wandb.

In [4]:
import wandb
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# 1. init wandb
wandb.init(project="classification-demo")

# 2. загрузка данных (без скачивания)
data = load_breast_cancer(as_frame=True)
df = data.frame

# 3. признаки и таргет
X = df.drop("target", axis=1)
y = df["target"]

# 4. train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# 5. модель
model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)

# 6. предсказания
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# 7. метрики
metrics = {
    "model": "LogisticRegression",
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_proba)
}

# 8. лог в wandb
wandb.log(metrics)

print(metrics)

{'model': 'LogisticRegression', 'accuracy': 0.9912280701754386, 'precision': 1.0, 'recall': 0.9861111111111112, 'f1': 0.993006993006993, 'roc_auc': 1.0}


# ## 3. Обучение Random Forest
Обучаю вторую модель и сравниваю результат.

In [3]:
import wandb
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

wandb.init(project="classification-demo", reinit=True)

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

rf_metrics = {
    "model": "RandomForest",
    "accuracy": accuracy_score(y_test, y_pred_rf),
    "precision": precision_score(y_test, y_pred_rf),
    "recall": recall_score(y_test, y_pred_rf),
    "f1": f1_score(y_test, y_pred_rf),
    "roc_auc": roc_auc_score(y_test, y_proba_rf)
}

wandb.log(rf_metrics)
print(rf_metrics)
wandb.finish()

accuracy,▁
f1,▁
precision,▁
recall,▁
roc_auc,▁
accuracy,0.94737
f1,0.96
precision,0.94737
recall,0.97297
roc_auc,0.99257


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


{'model': 'RandomForest', 'accuracy': 0.956140350877193, 'precision': 0.948051948051948, 'recall': 0.9864864864864865, 'f1': 0.9668874172185431, 'roc_auc': 0.9935810810810811}


accuracy,▁
f1,▁
precision,▁
recall,▁
roc_auc,▁
accuracy,0.95614
f1,0.96689
model,RandomForest
precision,0.94805
recall,0.98649
roc_auc,0.99358


# ## 4. Вывод
По результатам сравнения лучшей моделью оказалась Logistic Regression.